# 🔬 OSFT 파인튜닝

## OSFT (Orthogonal Subspace Fine-Tuning)란?

OSFT는 가중치 행렬을 직교 분해하여 선택적으로 학습하는 파인튜닝 기법입니다.

### LoRA와의 차이점

| 구분 | LoRA | OSFT |
|------|------|------|
| **접근 방식** | 저차원 어댑터 행렬 추가 | 가중치 직교 분해 후 선택적 해동 |
| **메모리** | 어댑터만큼 추가 메모리 | 분해 + 활성화 + 옵티마이저 메모리 |
| **파라미터** | rank, alpha | unfreeze_rank_ratio |
| **모듈성** | 어댑터 분리 가능 | 전체 모델에 반영됨 |
| **망각 저항** | 원본 보존으로 양호 | 직교 부분 공간으로 간섭 최소화 |

### OSFT의 특징
- 가중치 행렬 W를 직교 분해: W = U·Σ·V^T
- `unfreeze_rank_ratio`만큼의 특이값/벡터만 학습 대상으로 선택
- 나머지 직교 부분 공간은 고정 → 기존 지식 보존에 유리
- LoRA보다 더 많은 메모리를 사용하지만, 망각 저항이 우수할 수 있음

### 이 노트북의 설정
- **기본 모델**: `Qwen/Qwen3-4B-Instruct-2507` (LoRA와 동일)
- **unfreeze_rank_ratio**: 0.25 (전체 rank의 25% 학습)
- **데이터**: LoRA와 동일한 정규 데이터 (독립 학습)

> ⚠️ OSFT는 LoRA와 **독립적**으로 실행됩니다.  
> 동일한 기본 모델, 동일한 학습 데이터에서 시작합니다.  
> LoRA 체크포인트 위에 OSFT를 적용하지 **않습니다**.

In [ ]:
"""환경 부트스트랩 — local과 workbench 모두 지원."""

import subprocess, sys, os
from pathlib import Path

# 프로젝트 루트 탐색
_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

# 패키지 설치 확인 및 자동 설치
try:
    import rhoai_model_training_lab  # noqa: F401
    print("✅ rhoai_model_training_lab 패키지 확인됨")
except ImportError:
    print("📦 패키지 설치 중... (최초 1회)")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-e", str(_project_root),
             "--extra-index-url", "https://pypi.org/simple/"],
            stdout=subprocess.DEVNULL,
        )
        print("✅ 설치 완료")
    except subprocess.CalledProcessError:
        # Red Hat Workbench 등 제한된 환경: sys.path fallback
        print("⚠️  pip install 실패 — sys.path fallback 사용")
        _src = str(_project_root / "src")
        if _src not in sys.path:
            sys.path.insert(0, _src)
        # 핵심 의존성만 설치 시도
        for _dep in ["pydantic", "python-dotenv", "pyyaml", "rich", "httpx"]:
            try:
                __import__(_dep.replace("-", "_"))
            except ImportError:
                subprocess.call(
                    [sys.executable, "-m", "pip", "install", _dep,
                     "--extra-index-url", "https://pypi.org/simple/"],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                )
        import rhoai_model_training_lab  # noqa: F401
        print("✅ sys.path fallback 설정 완료")


In [ ]:
"""Load OSFT config and validate bundle compatibility."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_training_config, load_bundle_config, PROJECT_ROOT,
)
from rhoai_model_training_lab.data import BundleManager

load_env()

# Load OSFT config
osft_config = load_training_config("osft")
model_id = osft_config["model"]["model_id"]
model_revision = osft_config["model"]["model_revision"]

print(f"모델: {model_id} (rev: {model_revision})")
print(f"OSFT unfreeze_rank_ratio: {osft_config['osft']['unfreeze_rank_ratio']}")
print(f"시드: {osft_config['training']['seed']}")

# Load and validate bundle (same bundle as LoRA)
release_config = load_bundle_config()
bundle_base = release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")
bundle_path = PROJECT_ROOT / bundle_base

print(f"\n번들 경로: {bundle_path}")
mgr = BundleManager.load_bundle(bundle_path)
manifest = mgr.manifest

# Compatibility check (same base model as LoRA)
compat = mgr.validate_compatibility(model_id)
if compat.errors:
    for err in compat.errors:
        print(f"  ❌ {err}")
    raise RuntimeError(
        "번들 호환성 검증 실패 — 학습을 진행할 수 없습니다.\n"
        "data_preparation/ 노트북에서 올바른 번들을 생성하세요."
    )

print(f"\n✅ 번들 호환성 검증 통과")
print(f"   학습 샘플: {manifest.canonical_train_count} (LoRA와 동일)")
print(f"   검증 샘플: {manifest.canonical_validation_count}")
print(f"   번들 버전: {manifest.bundle_version}")

In [ ]:
"""Preview training data with OSFT-specific formatting."""

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Load OSFT-specific training data
train_data = mgr.get_training_samples("osft", "train")
val_data = mgr.get_training_samples("osft", "validation")

print(f"OSFT 학습 데이터: {len(train_data)} 샘플")
print(f"OSFT 검증 데이터: {len(val_data)} 샘플")

# Verify sample parity with LoRA
lora_train = mgr.get_training_samples("lora", "train")
print(f"\n📊 LoRA 학습 샘플 수: {len(lora_train)}")
print(f"   OSFT 학습 샘플 수: {len(train_data)}")
if len(train_data) == len(lora_train):
    print("   ✅ 샘플 수 일치 — 동일한 정규 데이터에서 변환")
else:
    print("   ⚠️  샘플 수 불일치 — 백엔드별 변환 차이 확인 필요")

# Preview with OSFT formatting focus
print("\n" + "=" * 70)
print("📝 OSFT 학습 데이터 미리보기")
print("=" * 70)

for i, sample in enumerate(train_data[:2]):
    messages = sample.get("messages", [])
    print(f"\n--- 샘플 {i+1} ---")
    print(f"메시지 수: {len(messages)}")

    for msg in messages:
        role = msg["role"]
        content = msg.get("content", "") or ""
        is_target = role == "assistant"
        mask_icon = "🎯 [학습 대상]" if is_target else "🚫 [마스킹됨]"

        display = content[:200] + "..." if len(content) > 200 else content
        print(f"\n  {mask_icon} [{role}]: {display}")

    # Token analysis
    try:
        tokens = tokenizer.apply_chat_template(messages, tokenize=True)
        print(f"\n  토큰 수: {len(tokens)}")
    except Exception:
        pass

# Memory estimation note
print(f"\n{'=' * 70}")
print("💾 OSFT 메모리 참고사항:")
print(f"  - unfreeze_rank_ratio: {osft_config['osft']['unfreeze_rank_ratio']}")
print("  - OSFT는 LoRA보다 더 많은 GPU 메모리를 사용합니다")
print("  - 분해(decomposition) + 활성화(activation) + 옵티마이저 상태 포함")
print(f"  - 배치 크기: {osft_config['training_args']['per_device_train_batch_size']}")
print(f"  - 기울기 누적: {osft_config['training_args']['gradient_accumulation_steps']}")

In [ ]:
"""Train using training_hub.osft — actual API call."""

import time
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU가 감지되지 않았습니다.\n"
        "OSFT 학습에는 CUDA 호환 GPU가 필요합니다.\n"
        "GPU가 있는 환경에서 이 노트북을 실행하세요."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")
print()

# Prepare arguments from config
train_file = str(PROJECT_ROOT / osft_config["data"]["train_file"])
val_file = str(PROJECT_ROOT / osft_config["data"]["validation_file"])
output_dir = str(PROJECT_ROOT / osft_config["training_args"]["output_dir"])

print("=" * 70)
print("🚀 OSFT 학습 시작")
print("=" * 70)
print(f"  학습 데이터: {train_file}")
print(f"  검증 데이터: {val_file}")
print(f"  출력 디렉토리: {output_dir}")
print(f"  에포크: {osft_config['training_args']['num_train_epochs']}")
print(f"  배치 크기: {osft_config['training_args']['per_device_train_batch_size']}")
print(f"  기울기 누적: {osft_config['training_args']['gradient_accumulation_steps']}")
print(f"  학습률: {osft_config['training_args']['learning_rate']}")
print(f"  unfreeze_rank_ratio: {osft_config['osft']['unfreeze_rank_ratio']}")
print()

start_time = time.time()

# Actual training_hub API call
from training_hub import osft as osft_train

training_result = osft_train(
    model_id=model_id,
    model_revision=model_revision,
    train_file=train_file,
    validation_file=val_file,
    output_dir=output_dir,
    unfreeze_rank_ratio=osft_config["osft"]["unfreeze_rank_ratio"],
    num_train_epochs=osft_config["training_args"]["num_train_epochs"],
    per_device_train_batch_size=osft_config["training_args"]["per_device_train_batch_size"],
    gradient_accumulation_steps=osft_config["training_args"]["gradient_accumulation_steps"],
    learning_rate=osft_config["training_args"]["learning_rate"],
    weight_decay=osft_config["training_args"]["weight_decay"],
    warmup_ratio=osft_config["training_args"]["warmup_ratio"],
    lr_scheduler_type=osft_config["training_args"]["lr_scheduler_type"],
    max_seq_length=osft_config["data"]["max_seq_length"],
    bf16=osft_config["training_args"]["bf16"],
    gradient_checkpointing=osft_config["training_args"]["gradient_checkpointing"],
    seed=osft_config["training"]["seed"],
    logging_steps=osft_config["training_args"]["logging_steps"],
    eval_steps=osft_config["training_args"]["eval_steps"],
    save_steps=osft_config["training_args"]["save_steps"],
    save_total_limit=osft_config["training_args"]["save_total_limit"],
    resume_from_checkpoint=osft_config["training_args"].get("resume_from_checkpoint", True),
)

wall_time = time.time() - start_time

print(f"\n✅ OSFT 학습 완료!")
print(f"  소요 시간: {wall_time/60:.1f}분")
print(f"  최종 학습 손실: {getattr(training_result, 'train_loss', 'N/A')}")
print(f"  최종 검증 손실: {getattr(training_result, 'eval_loss', 'N/A')}")

In [ ]:
"""Inspect results — loss curve and memory usage comparison."""

import json

# Load training logs
log_history = getattr(training_result, "log_history", None)
output_dir_path = Path(output_dir)

if log_history is None:
    state_path = output_dir_path / "trainer_state.json"
    if state_path.exists():
        with open(state_path) as f:
            state = json.load(f)
        log_history = state.get("log_history", [])

if log_history:
    train_steps = [e["step"] for e in log_history if "loss" in e]
    train_losses = [e["loss"] for e in log_history if "loss" in e]
    eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
    eval_losses = [e["eval_loss"] for e in log_history if "eval_loss" in e]

    try:
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Loss curve
        axes[0].plot(train_steps, train_losses, label="학습 손실", alpha=0.7)
        if eval_losses:
            axes[0].plot(eval_steps, eval_losses, label="검증 손실", marker="o", markersize=4)
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("OSFT 학습 손실 곡선")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Memory comparison with LoRA (if available)
        osft_peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
        lora_result_path = PROJECT_ROOT / "checkpoints" / "lora" / "training_result.json"
        lora_peak_vram = 0
        if lora_result_path.exists():
            with open(lora_result_path) as f:
                lora_data = json.load(f)
            lora_peak_vram = lora_data.get("peak_vram_gb", 0)

        methods = ["LoRA", "OSFT"]
        vrams = [lora_peak_vram, osft_peak_vram]
        colors = ["#2196F3", "#FF9800"]
        axes[1].bar(methods, vrams, color=colors)
        axes[1].set_ylabel("Peak VRAM (GB)")
        axes[1].set_title("피크 VRAM 사용량 비교")
        for i, v in enumerate(vrams):
            if v > 0:
                axes[1].text(i, v + 0.1, f"{v:.1f} GB", ha="center")

        plt.tight_layout()
        plt.show()
    except ImportError:
        osft_peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
        print("\nOSFT 학습 손실 추이:")
        for step, loss in zip(train_steps[-10:], train_losses[-10:]):
            bar = "█" * int(loss * 20)
            print(f"  Step {step:>6}: {loss:.4f} {bar}")

    # Summary
    osft_peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print("\n--- OSFT 학습 메트릭 요약 ---")
    if train_losses:
        print(f"  초기 손실: {train_losses[0]:.4f}")
        print(f"  최종 손실: {train_losses[-1]:.4f}")
    if eval_losses:
        print(f"  최종 검증 손실: {eval_losses[-1]:.4f}")
        print(f"  최소 검증 손실: {min(eval_losses):.4f}")
    print(f"  피크 VRAM: {osft_peak_vram:.1f} GB")
    print(f"  학습 시간: {wall_time/60:.1f}분")
else:
    osft_peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print("학습 로그를 찾을 수 없습니다.")

In [ ]:
"""Log OSFT results to MLflow."""

from rhoai_model_training_lab.schemas.training import TrainingResult

osft_result = TrainingResult(
    method="osft",
    model_id=model_id,
    model_revision=model_revision,
    bundle_id=manifest.bundle_name,
    bundle_hash=manifest.bundle_hash,
    seed=osft_config["training"]["seed"],
    train_samples=manifest.canonical_train_count,
    validation_samples=manifest.canonical_validation_count,
    total_steps=train_steps[-1] if train_steps else 0,
    final_train_loss=train_losses[-1] if train_losses else 0.0,
    final_eval_loss=eval_losses[-1] if eval_losses else None,
    best_eval_loss=min(eval_losses) if eval_losses else None,
    wall_time_seconds=wall_time,
    peak_vram_gb=osft_peak_vram,
    gpu_name=torch.cuda.get_device_name(0),
    checkpoint_path=output_dir,
)

mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
experiment_name = os.environ.get("MLFLOW_EXPERIMENT_TRAINING", "rhoai-model-training-lab-training")

if mlflow_uri:
    try:
        import mlflow

        mlflow.set_tracking_uri(mlflow_uri)
        mlflow.set_experiment(experiment_name)

        with mlflow.start_run(run_name=f"osft-{manifest.bundle_version}") as run:
            mlflow.log_param("method", "osft")
            mlflow.log_param("model_id", model_id)
            mlflow.log_param("bundle_id", manifest.bundle_name)
            mlflow.log_param("bundle_version", manifest.bundle_version)
            mlflow.log_param("unfreeze_rank_ratio", osft_config["osft"]["unfreeze_rank_ratio"])
            mlflow.log_param("learning_rate", osft_config["training_args"]["learning_rate"])
            mlflow.log_param("seed", osft_config["training"]["seed"])

            mlflow.log_metric("final_train_loss", osft_result.final_train_loss)
            if osft_result.final_eval_loss is not None:
                mlflow.log_metric("final_eval_loss", osft_result.final_eval_loss)
            if osft_result.best_eval_loss is not None:
                mlflow.log_metric("best_eval_loss", osft_result.best_eval_loss)
            mlflow.log_metric("wall_time_seconds", osft_result.wall_time_seconds)
            mlflow.log_metric("peak_vram_gb", osft_result.peak_vram_gb)
            mlflow.log_metric("total_steps", osft_result.total_steps)
            mlflow.log_metric("train_samples", osft_result.train_samples)

            osft_result.mlflow_run_id = run.info.run_id
            osft_result.mlflow_experiment = experiment_name
            print(f"✅ MLflow 기록 완료 (run_id: {run.info.run_id})")

    except Exception as exc:
        print(f"⚠️  MLflow 기록 실패: {exc}")
        print("  학습 결과는 로컬에 저장되었습니다.")
else:
    print("⚠️  MLFLOW_TRACKING_URI 미설정 — 로컬 결과만 저장합니다.")

# Save result locally
result_path = Path(output_dir) / "training_result.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w") as f:
    f.write(osft_result.model_dump_json(indent=2))
print(f"📄 학습 결과 저장: {result_path}")

In [ ]:
"""Verify OSFT checkpoint."""

from rich.table import Table
from rich.console import Console

console = Console()

checkpoint_dir = Path(output_dir)
checks = []

# Check for model files
has_model_files = (
    list(checkpoint_dir.glob("*.safetensors"))
    or list(checkpoint_dir.glob("*.bin"))
    or list(checkpoint_dir.glob("model*.safetensors"))
)
checks.append(("모델 가중치 파일", bool(has_model_files)))

# Check config
has_config = (checkpoint_dir / "config.json").exists()
checks.append(("모델 설정 파일", has_config))

# Check tokenizer
has_tokenizer = (checkpoint_dir / "tokenizer_config.json").exists()
checks.append(("토크나이저 파일", has_tokenizer))

# Try reloading
reload_ok = False
try:
    from transformers import AutoModelForCausalLM

    # Just verify the config can be loaded (don't load full model)
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(str(checkpoint_dir), trust_remote_code=True)
    reload_ok = True
    print(f"✅ 모델 설정 리로드 성공: {cfg.model_type}")
except Exception as exc:
    print(f"⚠️  모델 설정 리로드 실패: {exc}")

checks.append(("모델 리로드 검증", reload_ok))

# Checkpoint size
total_size = sum(f.stat().st_size for f in checkpoint_dir.rglob("*") if f.is_file())
size_gb = total_size / (1024**3)
checks.append((f"체크포인트 크기 ({size_gb:.2f} GB)", size_gb > 0))

# Summary table
table = Table(title="🔍 OSFT 체크포인트 검증", show_header=True)
table.add_column("항목", style="bold")
table.add_column("상태")

for name, ok in checks:
    status = "✅" if ok else "❌"
    table.add_row(name, status)

console.print(table)

all_ok = all(ok for _, ok in checks)
if all_ok:
    print("\n🎉 OSFT 학습이 성공적으로 완료되었습니다!")
    print("\n다음 단계:")
    print("  📓 05_export_and_deploy.ipynb — 모델 내보내기 및 배포")
    print("  LoRA와 OSFT 체크포인트가 모두 준비되었으므로 비교 평가가 가능합니다.")
else:
    print("\n⚠️  일부 검증 항목이 실패했습니다. 위 결과를 확인하세요.")